# Creating a Synthetic Dataset for SAE

This example generates a CSV dataset for SAE training and evaluation. The dataset contains texts for hidden-state extraction, concept labels for semantic-separability checks, and fields for later intervention tests.

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
openrouter_api_key = os.environ["OPENROUTER_API_KEY"]

from data_utils.dataset_creating import create_synthetic_dataset, split_dataset

This example reads `OPENROUTER_API_KEY` from the root `.env` file and passes it to `create_synthetic_dataset`. Generation is performed through the OpenRouter API with the `deepseek/deepseek-v4-flash` model.

In [ ]:
dataset_path = create_synthetic_dataset(
    dataset_theme="sae_experimental",
    samples_per_concept=50,
    output_dir=PROJECT_ROOT / "data",
    openrouter_api_key=openrouter_api_key,
    language="en",
    seed=42,
)

dataset_path

In [ ]:
import csv

with dataset_path.open("r", encoding="utf-8", newline="") as csv_file:
    rows = list(csv.DictReader(csv_file))

len(rows), rows[0]

The split is stratified by `concept_label` so that the train and test sets preserve the concept set needed to evaluate SAE feature separability.

In [ ]:
train_path, test_path = split_dataset(
    dataset_csv_path=dataset_path,
    test_size=0.2,
    seed=42,
    stratify_by="concept_label",
)

train_path, test_path

In [ ]:
def count_rows(path):
    with path.open("r", encoding="utf-8", newline="") as csv_file:
        return sum(1 for _ in csv.DictReader(csv_file))

count_rows(train_path), count_rows(test_path)

Next, the `text` field can be passed to `LLM.get_hidden_state` to extract activations, while `concept_label`, `target_behavior`, and `target_token` can be used for separability and intervention-selectivity calculations.